In [1]:
import math
import os
import pickle
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import reconstructing
import stats

sys.path.append(os.path.abspath(".."))

from initial_solution import jigs_initial_sequence


In [2]:
directory_path = r"path/to/initial_results"
project_root = os.path.abspath("..")


def resolve_existing_path(base_dir, candidates):
    for name in candidates:
        candidate = os.path.join(base_dir, name)
        if os.path.exists(candidate):
            return candidate
    raise FileNotFoundError(
        f"None of the expected files were found in {base_dir}: {candidates}"
    )


initial_solution_path = resolve_existing_path(
    directory_path,
    ["initial_solution.pkl", "intial_solution.pkl"],
)
initial_output_path = resolve_existing_path(
    directory_path,
    ["initial_output.pkl", "intial_output.pkl"],
)

print("Initial solution file:", initial_solution_path)
print("Initial output file:", initial_output_path)


Initial solution file: path/to/initial_results/initial_solution.pkl
Initial output file: path/to/initial_results/initial_output.pkl


In [14]:
with open(initial_solution_path, "rb") as f:
    initial_solution = pickle.load(f)

with open(initial_output_path, "rb") as f:
    initial_output = pickle.load(f)

initial_entry = {
    "solution": initial_solution,
    "result": initial_output,
    "objectives": np.array(
        [
            initial_output["labour_cost"],
            initial_output["total_tardiness"],
            initial_output["severity"],
        ],
        dtype=float,
    ),
}

print(type(initial_solution))
print(type(initial_output))
print("Initial solution keys:", initial_solution.keys())
print("Initial output keys:", initial_output.keys())
cost = initial_entry["objectives"][0]          
tardiness_hours = initial_entry["objectives"][1] / 3600
severity_hours = initial_entry["objectives"][2] / 3600

print("Initial Cost £", cost)
print("Initial Tardiness h", tardiness_hours)
print("Initial Severity h", severity_hours)


<class 'dict'>
<class 'dict'>
Initial solution keys: dict_keys(['job_sequence', 'staffing_plan', 'station_assignment'])
Initial output keys: dict_keys(['labour_cost', 'severity', 'total_tardiness', 'makespan', 'tardiness_vector', 'completion_times', 'station_assignment', 'assignments', 'station_day_utilization'])
Initial Cost £ 30720.0
Initial Tardiness h 176955.22208682497
Initial Severity h 139.0064588270424


In [4]:
data = pd.read_csv(os.path.join(project_root, "your_input_file.csv"))
data["deadline"] = pd.to_datetime(data["deadline"], dayfirst=True, errors="coerce")
start_date = data["deadline"].min() - pd.Timedelta(days=7)
data["DeadlineSeconds"] = (data["deadline"] - start_date).dt.total_seconds().astype(float)

print("Jobs in raw data:", len(data))
print("Start date:", start_date)


Jobs in raw data: 1769
Start date: 2025-05-05 00:00:00


notebook_cell.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data["deadline"] = pd.to_datetime(data["deadline"], dayfirst=True, errors="coerce")


In [5]:
multi_workers_efficiency = 0.9
jigs_capacity = 4
max_jigs = 4
staff_cost = 20
Day_work_time = 8 * 3600


In [9]:
job_sequence_init, staffing_plan_init, data_sorted_init = jigs_initial_sequence(
    data,
    start_date,
    efficiency_alpha=multi_workers_efficiency,
    T_max=Day_work_time,
    n_stations=max_jigs,
    active_stations=2,
    workers_per_active_station=2,
)

print("Jobs in sorted data:", len(data_sorted_init))
print("Saved job sequence length:", len(initial_entry["solution"]["job_sequence"]))
print("Saved staffing plan shape:", np.asarray(initial_entry["solution"]["staffing_plan"]).shape)


Jobs in sorted data: 1769
Saved job sequence length: 1769
Saved staffing plan shape: (94, 4)


In [10]:
initial_schedule = reconstructing.builder(
    entry=initial_entry,
    data=data_sorted_init,
    start_date=start_date,
    T_max=Day_work_time,
    efficiency_alpha=multi_workers_efficiency,
)

initial_schedule.to_csv("initial_schedule.csv", index=False)


In [11]:
time_stats_df, agg_delay = stats.time_stats(
    fs=initial_schedule,
    staffing_plan=initial_entry["solution"]["staffing_plan"],
    start_date=start_date,
    worker_hourly_cost=staff_cost,
    tardiness_vector=initial_entry["result"]["tardiness_vector"],
    T_max=Day_work_time,
)

time_stats_df.to_csv("initial_time_stats.csv", index=False)
agg_delay.to_csv("initial_agg_delay.csv", index=False)



In [12]:
station_day_stats, station_summary = stats.split_jobs_by_day(
    fs=initial_schedule,
    day_work_time=Day_work_time,
)

station_day_stats.to_csv("initial_station_day_stats.csv", index=False)
station_summary.to_csv("initial_station_summary.csv", index=False)

In [13]:
makespan_sec = float(initial_entry["result"]["makespan"])
makespan_days = math.ceil(makespan_sec / (24 * 3600))

print("Makespan [sec]:", makespan_sec)
print("Makespan [days]:", makespan_days)

with open("initial_makespan_days.txt", "w") as f:
    f.write(f"Makespan (seconds): {makespan_sec}\n")
    f.write(f"Makespan (days): {makespan_days}\n")


Makespan [sec]: 5631644.187063238
Makespan [days]: 66
